In [1]:
import pandas as pd
import os
from pathlib import Path
import shutil
import re
import unicodedata

Crear directorio `csv_cleaned`

In [3]:
try:
    root = Path("/home/daniel/code/clasificador_selectividad")
    os.chdir(root)
except Exception as e:
    root = Path("/Users/daniel/code/clasificador_selectividad")
    os.chdir(root)

In [37]:
Path(root / "csv_cleaned").mkdir(exist_ok=True)

Cargar archivos csv

In [44]:
df = pd.read_csv(root / "csv_enunciados/ejercicios_enunciados.csv", encoding="utf-8")

In [48]:
def create_id(row):
    id_string = f"{row["asignatura"]}_{row["año"]}_{row["convocatoria"]}"
    if not pd.notna(row["ejercicio"]):
        row["ejercicio"] = row["ejercicio"].replace(",", "").replace(" ", "_")
        id_string += f"_{row["ejercicio"]}"
    return id_string.lower().replace(" ", "_")

In [41]:
def clean_dataframe(df):
    df_sin_metadatos = df[df.aviso == "sin metadatos"]
    df = df[df.aviso.isna()]

    # modify ejercicio column
    df.ejercicio = df.ejercicio.apply(lambda x : x.replace("Opción", "opción").replace(".", ",") if not pd.isna(x) else x)
    
    # create id
    df["id"] = df.apply(lambda row: create_id(row), axis = 1)
    
    # rearrange columns
    df = df[["id"] + list(df.columns[:-2])]

    # remove gravity from physics statements
    patron = r"g\s*=\s*9[,.]8\s*m\s*[·.]?\s*s\s*-\s*2\s*"
    mask = df.asignatura == "Física"
    df.loc[mask, "enunciado"] = df.loc[mask, "enunciado"].str.replace(patron, "", regex=True)
    
    # normalize text
    df.enunciado = df.enunciado.apply(lambda texto : unicodedata.normalize("NFKC", texto) if not pd.isna(texto) else texto)

    # trim multiple spaces
    df.enunciado = df.enunciado.apply(lambda texto : re.sub(r"[ \t]+", " ", texto).strip() if not pd.isna(texto) else texto)

    
    
    # export clean csv
    output_dir = Path(root / "csv_cleaned")
    output_dir.mkdir(exist_ok=True)
    
    df.to_csv(output_dir / "enunciados_ejercicios_clean.csv", encoding="utf-8", index = False)

In [46]:
clean_dataframe(df)

In [46]:
df = pd.read_csv(root / "csv_cleaned/enunciados_ejercicios_clean.csv", encoding="utf-8")

In [51]:
df.sample(3)

,id,asignatura,asignatura_carpeta,año,convocatoria,ejercicio,tema,info_ejercicio,enunciado,archivo
276,física_2021_reserva_4,Física,Física,2021,Reserva 4,Ejercicio C1,Ondas,FISICA. 2021. RESERVA 4. EJERCICIO C1,a) i) Explique brevemente qué es una onda elec...,2021 - Ondas.pdf
3785,química_2011_reserva_2,Química,Química,2011,Reserva 2,"Ejercicio 4, opción B",Ácido Base,QUÍMICA. 2011. RESERVA 2. EJERCICIO 4. OPCIÓN B,Complete los siguientes equilibrios ácido-base...,2011 - Ácido Base.pdf
2885,mates_ii_2024_reserva_4,Mates II,Mates II,2024,Reserva 4,Ejercicio 8,Geometría,MATEMÁTICAS II. 2024. RESERVA 4. EJERCICIO 8,Sea \n1\nel plano determinado por los puntos ...,2024 - Geometría.pdf


In [50]:
df.id = df.apply(create_id, axis = 1)

In [52]:
df.to_csv(root/"csv_cleaned/enunciados_ejercicios_clean.csv", index = False, encoding="utf-8")